In [27]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

"""
    def kısmı fonksiyon olarak istediğimiz hissenin sembolünü alır ve hisseyi sunucudan çeker 
    işlenebilir hale getirir
    """
def veri_getir(sembol):
    """
    hisse hisse verisini çekmek için gerekli yolu tanımlar yf.Ticker kodu y financedeki bir
    fonksiyondur ve sembol ile gelen hisseyi alma yolu nesnesini oluşmasını sağlar.Ticker
    ayrıca hisse nin kısaltışmış adı demektir.Veri nesnesi ise history fonksiyonuyla 
    period= ile ne süre istenirse o kadar sürelik geçmişe dönük veri alır.Y harfi yıl
    D harfi gün M harfide ay için kullanılır önündeki sayıda kaç ay yıl onu belirtir.
    Eğer max yazılırsa ilk halka arz olduğu zamana gider.
    """
    hisse = yf.Ticker(sembol)
    veri = hisse.history(period="5y")
    
    # 1. KONTROL: Hiç veri var mı? (Hisse kodu yanlış olabilir)
    if veri.empty:
        print(f"❌ HATA: '{sembol}' diye bir hisse bulunamadı veya veri yok!")
        return None
    
    # 2. BİLGİLENDİRME: Veri hangi tarihten başlıyor?
    # veri.index[0] -> Tablonun ilk satırının tarihi (Başlangıç)
    # .strftime(...) -> Tarihi güzel okunan bir formata çevirir (Gün-Ay-Yıl)
    baslangic_tarihi = veri.index[0].strftime('%d-%m-%Y')
    
    
    
    return veri


In [28]:
"""ilk veri getir fonksiyonundan çıkan veriyi alıp işler SMA20 ve SMA50 yi hesaplar başka değerleride 
hesaplamak isterse buraya ekleyebiliriz.Verinin kopyasını oluşturuyoruz bozulmaması için kullanırız.
dfSMA20 ile kısa vadeli değeri hesapları df close kısmı kapanış değerlerinin son 20 sini alıp 20 ye
bölüp değeri bulur.SMA50 de aynı mantığı orta vadeli hesaplar.Eğer SMA20 aşağıdan yukarı SMA50 yi 
keserse hisse yükselebilir tam tersi olursa hisse düşer manasına gelir.
"""
def hesapla(gelen_veri):
    df = gelen_veri.copy()
    df['SMA20'] = df['Close'].rolling(window=20).mean()
    df['SMA50'] = df['Close'].rolling(window=50).mean()

    return df


In [32]:
# --- 3. FABRİKA: GÖRSELLEŞTİRME (BOYAMA) ---
def ciz(veri, sembol):
    """
    Veriyi alır, Mum grafiği ve SMA çizgilerini çizer.
    Gece/Gündüz modu ve zaman butonlarını ekler.
    """
    print("🎨 Grafik hazırlanıyor...")
    
    # 1. TUVALİ OLUŞTUR
    fig = go.Figure()

    # 2. KATMAN: MUMLAR (CANDLESTICK)
    fig.add_trace(go.Candlestick(
        x=veri.index,
        open=veri['Open'],
        high=veri['High'],
        low=veri['Low'],
        close=veri['Close'],
        name=f"{sembol} Fiyat"
    ))

    # 3. KATMAN: SMA 20 (MAVİ ÇİZGİ)
    # Eğer hesaplanmışsa çiz (Hata olmasın diye kontrol edebiliriz ama şimdilik gerek yok)
    fig.add_trace(go.Scatter(
        x=veri.index,
        y=veri['SMA20'],
        mode='lines',
        name='SMA 20 (Kısa Vade)',
        line=dict(color='cyan', width=1.5)
    ))

    # 4. KATMAN: SMA 50 (SARI ÇİZGİ)
    fig.add_trace(go.Scatter(
        x=veri.index,
        y=veri['SMA50'],
        mode='lines',
        name='SMA 50 (Orta Vade)',
        line=dict(color='yellow', width=1.5)
    ))

    # 5. AYARLAR (LAYOUT)
    fig.update_layout(
        title=f'{sembol} Teknik Analiz Grafiği',
        yaxis_title='Fiyat (TL)',
        
        # --- BAŞLANGIÇ AYARLARI ---
        template='plotly_dark',
        font=dict(color="white"),
        plot_bgcolor='#1f2630',
        paper_bgcolor='#1f2630',
        
        # --- HAFTASONU KESİCİ ---
        xaxis=dict(
            rangebreaks=[dict(bounds=["sat", "mon"])],
            rangeslider=dict(visible=False), # Alttaki çubuğu gizle
            type="date",
            
            # Zaman Butonları (1 Ay, 1 Yıl...)
            rangeselector=dict(
                bgcolor='rgba(0,0,0,0)', # Şeffaf
                buttons=list([
                    dict(count=1, label="1 Ay", step="month", stepmode="backward"),
                    dict(count=6, label="6 Ay", step="month", stepmode="backward"),
                    dict(count=1, label="1 Yıl", step="year", stepmode="backward"),
                    dict(count=3, label="3 Yıl", step="year", stepmode="backward"),
                    dict(step="all", label="Tümü")
                ])
            )
        ),
        
        # --- TEMA BUTONLARI (GECE / GÜNDÜZ) ---
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                x=0.5, y=1.15,
                showactive=False, # Seçili olanı boyama
                bgcolor='rgba(0,0,0,0)', # Şeffaf
                
                buttons=list([
                    dict(label="🌙 Gece", method="relayout", args=[{
                        "template": "plotly_dark",
                        "plot_bgcolor": "#1f2630", "paper_bgcolor": "#1f2630",
                        "font.color": "white",
                        "xaxis.rangeselector.font.color": "white"
                    }]),
                    dict(label="☀️ Gündüz", method="relayout", args=[{
                        "template": "plotly_white",
                        "plot_bgcolor": "white", "paper_bgcolor": "white",
                        "font.color": "black",
                        "xaxis.rangeselector.font.color": "black"
                    }])
                ])
            )
        ]
    )

    # 6. SAHNE!
    fig.show()

In [33]:
# --- ANA KONTROL MERKEZİ ---

hisse_adi = "THYAO.IS" # İstersen burayı GARAN.IS, ASELS.IS yapıp dene

# 1. Adım: Veriyi Getir
ham_veri = veri_getir(hisse_adi)

# 2. Adım: Veriyi Hesapla (Eğer veri geldiyse)
if ham_veri is not None:
    hazir_veri = hesapla(ham_veri)
    
    # 3. Adım: Çiz
    ciz(hazir_veri, hisse_adi)
else:
    print("İşlem iptal edildi.")

🎨 Grafik hazırlanıyor...


In [34]:
import time # Uyuma komutu için gerekli kütüphane

# --- SERİ ÜRETİM MERKEZİ ---

# 1. Analiz etmek istediğimiz hisselerin listesi
# İstediğin hisseyi buraya ekleyip çıkarabilirsin.
target_hisseler = [
    "THYAO.IS", # Türk Hava Yolları
    "GARAN.IS", # Garanti Bankası
    "ASELS.IS", # Aselsan
    "SISE.IS",  # Şişecam
    "AKBNK.IS", # Akbank
    "KCHOL.IS", # Koç Holding
    "EREGL.IS", # Ereğli Demir Çelik
    "SAHOL.IS", # Sabancı Holding
    "TUPRS.IS", # Tüpraş
    "BIMAS.IS"  # BİM
]

print(f"🚀 Toplam {len(target_hisseler)} hisse için analiz başlatılıyor...\n")

# 2. Döngü Başlıyor (Fabrika Bant Sistemi)
for sembol in target_hisseler:
    
    
    # A. Veriyi Getir
    ham_veri = veri_getir(sembol)
    
    # Veri başarıyla geldiyse devam et
    if ham_veri is not None:
        
        # B. Hesapla (Matematik)
        islenmis_veri = hesapla(ham_veri)
        
        # C. Çiz (Ressam)
        ciz(islenmis_veri, sembol)
        
       
    else:
        print(f"⚠️ {sembol} atlandı (Veri yok).")
    
    # 3. Dinlenme Molası (Yahoo bizi engellemesin diye)
    
    time.sleep(2) 
    print("-" * 50) # Araya çizgi çekelim karışmasın



🚀 Toplam 10 hisse için analiz başlatılıyor...

🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


...

--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
🎨 Grafik hazırlanıyor...


--------------------------------------------------
